# kNN and Naive Bayes

Implement (a) a k-Nearest Neighbours classifier from scratch using torch pairwise distances and majority vote, and (b) a Gaussian Naive Bayes classifier from scratch using per-class Gaussian log-likelihoods. Validate both against scikit-learn on Iris.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()` — never hardcoded. On Apple Silicon this runs on mps.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Imports and data

In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

SEED = 42
K = 5  # neighbours

iris = load_iris()
X, y = iris.data, iris.target

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

# Also keep torch tensors on device for the kNN from-scratch implementation
X_train = torch.tensor(X_train_np, dtype=torch.float32, device=device)
X_test  = torch.tensor(X_test_np,  dtype=torch.float32, device=device)
y_train = torch.tensor(y_train_np, dtype=torch.long,    device=device)
y_test  = torch.tensor(y_test_np,  dtype=torch.long,    device=device)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Classes: {iris.target_names}")

Train: torch.Size([105, 4])  Test: torch.Size([45, 4])
Classes: ['setosa' 'versicolor' 'virginica']


## k-Nearest Neighbours — theory

For a query point $\mathbf{x}$, find the $k$ training points with smallest Euclidean distance:

$$d(\mathbf{x}, \mathbf{x}_i) = \|\mathbf{x} - \mathbf{x}_i\|_2$$

Predict the **majority class** among the $k$ neighbours (ties broken by smallest label index).

**Key properties**
- Non-parametric: no model is fitted; the training set *is* the model.
- Lazy learner: zero training cost, $O(n \cdot d)$ prediction cost per query.
- Scale-sensitive: features with large ranges dominate Euclidean distance; always standardise.
- Large $k$ smooths the boundary (less variance, more bias); small $k$ is noisy.

In [3]:
def knn_predict(
    X_tr: torch.Tensor,
    y_tr: torch.Tensor,
    X_te: torch.Tensor,
    k: int,
) -> torch.Tensor:
    """k-NN classifier using vectorised torch pairwise L2 distances.

    Args:
        X_tr: (n_train, d) training features.
        y_tr: (n_train,) integer class labels.
        X_te: (n_test,  d) test features.
        k:    number of nearest neighbours.

    Returns:
        (n_test,) predicted class labels.
    """
    # Pairwise squared L2: ||x_i - x_j||^2 = ||x_i||^2 + ||x_j||^2 - 2 x_i·x_j
    # Shape: (n_test, n_train)
    sq_te = (X_te ** 2).sum(dim=1, keepdim=True)      # (n_test, 1)
    sq_tr = (X_tr ** 2).sum(dim=1, keepdim=True).T    # (1, n_train)
    dot   = X_te @ X_tr.T                             # (n_test, n_train)
    dist2 = sq_te + sq_tr - 2.0 * dot                 # (n_test, n_train)
    # Clamp tiny negatives from floating-point arithmetic
    dist2 = dist2.clamp(min=0.0)

    # Top-k nearest (smallest distance)
    _, nn_indices = dist2.topk(k, dim=1, largest=False)  # (n_test, k)
    nn_labels = y_tr[nn_indices]                          # (n_test, k)

    # Majority vote
    n_test = X_te.shape[0]
    n_classes = int(y_tr.max().item()) + 1
    votes = torch.zeros(n_test, n_classes, device=X_te.device, dtype=torch.long)
    for i in range(n_test):
        for lbl in nn_labels[i]:
            votes[i, lbl] += 1
    predictions = votes.argmax(dim=1)
    return predictions


preds_knn_scratch = knn_predict(X_train, y_train, X_test, k=K)
acc_knn_scratch = float((preds_knn_scratch == y_test).float().mean().item())
print(f"From-scratch k-NN (k={K}) accuracy: {acc_knn_scratch:.4f}")

From-scratch k-NN (k=5) accuracy: 0.9778


## Validation: k-NN vs sklearn

In [4]:
sk_knn = KNeighborsClassifier(n_neighbors=K, metric="euclidean")
sk_knn.fit(X_train_np, y_train_np)
preds_knn_sk = sk_knn.predict(X_test_np)
acc_knn_sklearn = float(np.mean(preds_knn_sk == y_test_np))

print(f"sklearn  KNeighborsClassifier accuracy: {acc_knn_sklearn:.4f}")
print(f"from-scratch k-NN accuracy:             {acc_knn_scratch:.4f}")

TOL_KNN = 0.05
diff_knn = abs(acc_knn_scratch - acc_knn_sklearn)
print(f"Difference: {diff_knn:.4f}  (tolerance {TOL_KNN})")
assert diff_knn < TOL_KNN, (
    f"k-NN accuracy gap too large: scratch={acc_knn_scratch:.4f}, "
    f"sklearn={acc_knn_sklearn:.4f}, diff={diff_knn:.4f}"
)
print("✓  Assertion passed: from-scratch k-NN matches sklearn within tolerance.")

sklearn  KNeighborsClassifier accuracy: 0.9778
from-scratch k-NN accuracy:             0.9778
Difference: 0.0000  (tolerance 0.05)
✓  Assertion passed: from-scratch k-NN matches sklearn within tolerance.


## Scale sensitivity demo (Exercise 2)

Two features with very different scales make Euclidean distance meaningless unless we standardise first. On the Iris dataset we can simulate this by inflating a single feature.

In [5]:
from sklearn.preprocessing import StandardScaler

# Inflate feature 1 (sepal width) 100x to simulate a 'dollars' scale
X_skewed = X_train_np.copy()
X_skewed[:, 1] *= 100.0
X_test_skewed = X_test_np.copy()
X_test_skewed[:, 1] *= 100.0

sk_knn_skewed = KNeighborsClassifier(n_neighbors=K, metric="euclidean")
sk_knn_skewed.fit(X_skewed, y_train_np)
acc_skewed = float(np.mean(sk_knn_skewed.predict(X_test_skewed) == y_test_np))

scaler = StandardScaler()
X_scaled_tr = scaler.fit_transform(X_skewed)
X_scaled_te = scaler.transform(X_test_skewed)
sk_knn_scaled = KNeighborsClassifier(n_neighbors=K, metric="euclidean")
sk_knn_scaled.fit(X_scaled_tr, y_train_np)
acc_scaled = float(np.mean(sk_knn_scaled.predict(X_scaled_te) == y_test_np))

print(f"k-NN on skewed (unscaled) features:  {acc_skewed:.4f}")
print(f"k-NN after StandardScaler:           {acc_scaled:.4f}")
print("→ Scaling rescues performance when a feature dominates by magnitude.")

k-NN on skewed (unscaled) features:  0.6667
k-NN after StandardScaler:           0.9111
→ Scaling rescues performance when a feature dominates by magnitude.


## Gaussian Naive Bayes — theory

Naive Bayes applies Bayes' rule:

$$\hat{y} = \arg\max_y\; p(y) \prod_j p(x_j \mid y)$$

The **conditional independence** (naïve) assumption factorises the likelihood across features.  Even when this assumption is violated, the model often classifies correctly because the argmax is preserved unless error accumulates badly.

**Gaussian Naive Bayes** models each $p(x_j \mid y)$ as a Gaussian with per-class mean and variance estimated from training data:

$$p(x_j \mid y) = \frac{1}{\sqrt{2\pi\sigma_{jy}^2}} \exp\!\left(-\frac{(x_j - \mu_{jy})^2}{2\sigma_{jy}^2}\right)$$

Taking logs avoids underflow from multiplying many small probabilities:

$$\log p(y \mid \mathbf{x}) \propto \log p(y) + \sum_j \log p(x_j \mid y)$$

In [6]:
class GaussianNaiveBayes:
    """Gaussian Naive Bayes classifier from scratch.

    Uses per-class Gaussian likelihoods with a small variance floor
    to avoid log(0) when a feature is constant inside a class.
    """

    _EPS: float = 1e-9  # variance floor

    def __init__(self) -> None:
        self._classes: np.ndarray | None = None
        self._log_priors: np.ndarray | None = None  # (n_classes,)
        self._means: np.ndarray | None = None       # (n_classes, n_features)
        self._vars: np.ndarray | None = None        # (n_classes, n_features)

    def fit(self, X: np.ndarray, y: np.ndarray) -> "GaussianNaiveBayes":
        self._classes = np.unique(y)
        n_samples, n_features = X.shape
        n_classes = len(self._classes)

        means = np.zeros((n_classes, n_features))
        vars_ = np.zeros((n_classes, n_features))
        log_priors = np.zeros(n_classes)

        for idx, cls in enumerate(self._classes):
            mask = y == cls
            X_cls = X[mask]
            means[idx] = X_cls.mean(axis=0)
            vars_[idx] = X_cls.var(axis=0) + self._EPS
            log_priors[idx] = np.log(mask.sum() / n_samples)

        self._means = means
        self._vars = vars_
        self._log_priors = log_priors
        return self

    def _log_likelihood(self, X: np.ndarray) -> np.ndarray:
        """Log-likelihood matrix of shape (n_samples, n_classes)."""
        n_samples = X.shape[0]
        n_classes = len(self._classes)
        ll = np.zeros((n_samples, n_classes))
        for idx in range(n_classes):
            mu = self._means[idx]        # (n_features,)
            var = self._vars[idx]        # (n_features,)
            # log N(x; mu, var) per feature, summed
            log_norm = -0.5 * np.log(2.0 * np.pi * var)
            log_exp  = -0.5 * ((X - mu) ** 2) / var
            ll[:, idx] = (log_norm + log_exp).sum(axis=1)
        return ll

    def predict(self, X: np.ndarray) -> np.ndarray:
        if self._classes is None:
            raise RuntimeError("Call fit() first.")
        log_posterior = self._log_likelihood(X) + self._log_priors  # broadcast
        return self._classes[np.argmax(log_posterior, axis=1)]

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Return class probabilities via softmax of log-posteriors."""
        lp = self._log_likelihood(X) + self._log_priors
        # Numerically stable softmax
        lp -= lp.max(axis=1, keepdims=True)
        exp_lp = np.exp(lp)
        return exp_lp / exp_lp.sum(axis=1, keepdims=True)


gnb = GaussianNaiveBayes()
gnb.fit(X_train_np, y_train_np)
preds_gnb_scratch = gnb.predict(X_test_np)
acc_gnb_scratch = float(np.mean(preds_gnb_scratch == y_test_np))
print(f"From-scratch GNB accuracy: {acc_gnb_scratch:.4f}")

From-scratch GNB accuracy: 0.9111


## Validation: Gaussian Naive Bayes vs sklearn

In [7]:
sk_gnb = GaussianNB()
sk_gnb.fit(X_train_np, y_train_np)
preds_gnb_sk = sk_gnb.predict(X_test_np)
acc_gnb_sklearn = float(np.mean(preds_gnb_sk == y_test_np))

print(f"sklearn  GaussianNB accuracy:    {acc_gnb_sklearn:.4f}")
print(f"from-scratch GNB accuracy:       {acc_gnb_scratch:.4f}")

TOL_GNB = 0.05
diff_gnb = abs(acc_gnb_scratch - acc_gnb_sklearn)
print(f"Difference: {diff_gnb:.4f}  (tolerance {TOL_GNB})")
assert diff_gnb < TOL_GNB, (
    f"GNB accuracy gap too large: scratch={acc_gnb_scratch:.4f}, "
    f"sklearn={acc_gnb_sklearn:.4f}, diff={diff_gnb:.4f}"
)
print("✓  Assertion passed: from-scratch GNB matches sklearn within tolerance.")

sklearn  GaussianNB accuracy:    0.9111
from-scratch GNB accuracy:       0.9111
Difference: 0.0000  (tolerance 0.05)
✓  Assertion passed: from-scratch GNB matches sklearn within tolerance.


## Laplace smoothing — worked example (Exercise 3)

For Multinomial Naive Bayes (text classification):

$$p(\text{token}_j \mid y) = \frac{\text{count}_{j,y} + \alpha}{\text{total\_count}_y + \alpha \cdot V}$$

This prevents zero probabilities for unseen tokens.

> **Note:** Laplace (add-one) smoothing applies to *Multinomial* NB, which models count data (e.g., word frequencies in text). The Gaussian NB implemented here does not use count smoothing; instead it adds a small variance floor (`var + eps`) to each per-class feature variance for numerical stability.

In [8]:
# Exercise 3: p("free" | spam) with Laplace smoothing
count_free_spam  = 8
total_count_spam = 100
vocab_size       = 50
alpha            = 1

p_free_spam = (count_free_spam + alpha) / (total_count_spam + alpha * vocab_size)
print(f'p("free" | spam) = ({count_free_spam} + {alpha}) / ({total_count_spam} + {alpha}*{vocab_size})')
print(f'                 = {count_free_spam + alpha} / {total_count_spam + alpha * vocab_size}')
print(f'                 = {p_free_spam:.6f}  (~{p_free_spam:.4f})')

expected = 9 / 150
assert abs(p_free_spam - expected) < 1e-10
print("✓  Laplace-smoothed probability matches expected value (9/150).")

p("free" | spam) = (8 + 1) / (100 + 1*50)
                 = 9 / 150
                 = 0.060000  (~0.0600)
✓  Laplace-smoothed probability matches expected value (9/150).


## Decision boundaries: k-NN vs Gaussian Naive Bayes (features 2 & 3)

In [9]:
feat_a, feat_b = 2, 3  # petal length, petal width

X_2d_tr = X_train_np[:, [feat_a, feat_b]]
X_2d_te = X_test_np[:,  [feat_a, feat_b]]

# Fit both classifiers on 2-D data
knn2d = KNeighborsClassifier(n_neighbors=K, metric="euclidean")
knn2d.fit(X_2d_tr, y_train_np)

gnb2d = GaussianNaiveBayes()
gnb2d.fit(X_2d_tr, y_train_np)

h = 0.02
x_min, x_max = X_2d_tr[:, 0].min() - 0.5, X_2d_tr[:, 0].max() + 0.5
y_min, y_max = X_2d_tr[:, 1].min() - 0.5, X_2d_tr[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
mesh = np.c_[xx.ravel(), yy.ravel()]

Z_knn = knn2d.predict(mesh).reshape(xx.shape)
Z_gnb = gnb2d.predict(mesh).reshape(xx.shape)

colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
cmap = plt.cm.RdYlBu

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (Z, title) in zip(
    axes,
    [(Z_knn, f"k-NN (k={K})"), (Z_gnb, "Gaussian Naive Bayes")],
):
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap)
    for cls, color in zip(np.unique(y), colors):
        mask = y_test_np == cls
        ax.scatter(
            X_2d_te[mask, 0], X_2d_te[mask, 1],
            c=color, label=iris.target_names[cls], edgecolors="k", s=40,
        )
    ax.set_xlabel(iris.feature_names[feat_a])
    ax.set_ylabel(iris.feature_names[feat_b])
    ax.set_title(title)
    ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("knn_gnb_boundary.png", dpi=80, bbox_inches="tight")
plt.show()
print("Plot saved to knn_gnb_boundary.png")

Plot saved to knn_gnb_boundary.png


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_71238/1548716458.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Idiomatic sklearn usage

In [10]:
# Idiomatic sklearn k-NN and Gaussian Naive Bayes
sk_knn_final = KNeighborsClassifier(n_neighbors=K, metric="euclidean")
sk_knn_final.fit(X_train_np, y_train_np)

sk_gnb_final = GaussianNB()
sk_gnb_final.fit(X_train_np, y_train_np)

print(f"sklearn KNeighborsClassifier (k={K}):  {float(np.mean(sk_knn_final.predict(X_test_np) == y_test_np)):.4f}")
print(f"sklearn GaussianNB:                    {float(np.mean(sk_gnb_final.predict(X_test_np) == y_test_np)):.4f}")
print()
print("Log-probabilities from GaussianNB (first 3 test samples):")
print(sk_gnb_final.predict_log_proba(X_test_np[:3]).round(3))

sklearn KNeighborsClassifier (k=5):  0.9778
sklearn GaussianNB:                    0.9111

Log-probabilities from GaussianNB (first 3 test samples):
[[-6.58746e+02 -1.47070e+01 -0.00000e+00]
 [-3.01967e+02 -5.00000e-03 -5.40500e+00]
 [-3.75692e+02 -1.66000e-01 -1.87800e+00]]


## Takeaways

### k-Nearest Neighbours
- **No training step** — the model stores training data. All cost is at prediction time: $O(n \cdot d)$ per query naively.
- **Distance = similarity proxy**: Euclidean, cosine, Manhattan are common. Feature scaling is non-negotiable for Euclidean kNN.
- **k controls bias-variance tradeoff**: $k=1$ has zero training error but high variance; large $k$ smooths boundaries.
- Best for: small datasets, embedding spaces where neighbour structure is meaningful, sanity-check baselines.

### Gaussian Naive Bayes
- **Conditional independence assumption**: $p(\mathbf{x} \mid y) = \prod_j p(x_j \mid y)$. Rarely exactly true, but sufficient for a useful classifier.
- **Log-sum instead of multiply**: avoids floating-point underflow when multiplying many small probabilities.
- **Gaussian likelihood** is appropriate for continuous features; Multinomial/Bernoulli variants suit text counts.
- **Laplace smoothing** ($\alpha > 0$) regularises Multinomial NB and prevents zero-probability tokens.
- Best for: fast baselines, high-dimensional sparse data (text), low-data regimes.

### When to choose which
| Situation | Prefer |
|-----------|--------|
| Small dataset, continuous features | **k-NN** |
| Text / sparse count features | **Naive Bayes** |
| Many training examples, slow prediction unacceptable | **Naive Bayes** |
| Features strongly correlated within class | **k-NN** (NB independence assumption breaks) |
| Need probabilistic outputs (calibrated) | **Naive Bayes** (but calibrate separately) |